<a href="https://colab.research.google.com/github/jagteshwar13/jagteshwar13/blob/main/Jagteshwar_EPAT_Statistical_Arbitrage_Uranium_CCJ_UEC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistical Arbitrage on Uranium Equities
## A Cointegration-Based Pairs Trading Study of Cameco (CCJ) and Uranium Energy Corp (UEC)

**EPAT Project: Statistical Arbitrage Strategy**

| | |
|---|---|
| **Author** | Jagteshwar Singh |
| **Programme** | Executive Programme in Algorithmic Trading (EPAT), QuantInsti |
| **Strategy** | Statistical arbitrage (pairs trading, mean reversion of a cointegrated spread) |
| **Data** | Barchart daily prices, January 2015 to December 2024 |
| **Tools** | Python (pandas, NumPy, statsmodels, Matplotlib) |

## 1. Abstract

This project asks whether two uranium equities, Cameco (CCJ) and Uranium Energy Corp (UEC), can be traded as a statistical-arbitrage pair. Daily prices for six uranium-related tickers and two non-uranium control tickers are screened with the Engle-Granger cointegration test. CCJ/UEC is selected on both economic and statistical grounds. A hedge ratio is estimated by OLS on the first 100 trading days, the resulting spread is tested for stationarity with the Augmented Dickey-Fuller (ADF) test, and a rolling mean-reversion rule is backtested. Performance is measured by cumulative return, Sharpe ratio and maximum drawdown, and then stress-tested for transaction costs, band construction and hedge-ratio stability with a walk-forward test. The figures quoted in the findings (Section 9) are generated by the code in this notebook.

## 2. Introduction

Statistical arbitrage exploits temporary price divergences between instruments whose prices are statistically related. In pairs trading two securities are combined in a hedged ratio so that their common risk factor cancels. If the resulting **spread** is stationary (the two prices are *cointegrated*), deviations from its mean should revert, and a position that is short the expensive leg and long the cheap leg earns that reversion.

Uranium equities are a natural place to look. Producers and developers are exposed to the same commodity and respond to the same news, yet their prices differ because of size, jurisdiction, production profile and investor sentiment. CCJ is the largest listed producer, while UEC is a US-focused company with a very different volatility profile, so their relative pricing is a candidate for a hedged mean-reversion trade.

## 3. Objectives

1. Collect daily prices for uranium equities and the sector ETF, plus two non-uranium control names (ARKK, ZM).
2. Screen every pair for correlation and cointegration and choose a pair on statistical **and** economic grounds.
3. Estimate the hedge ratio by OLS, build the spread and test it for stationarity.
4. Design and backtest a mean-reversion rule on the spread.
5. Evaluate performance (cumulative return, Sharpe ratio, drawdown) and stress-test the result.

## 4. Methodology

| Step | Method |
|---|---|
| Pair screen | Engle-Granger: regress one price on the other (with intercept), ADF test on the residual; both directions tested, stronger statistic kept |
| Hedge ratio | OLS of CCJ on UEC **without intercept** over the first 100 observations, then held fixed |
| Spread | $S_t = CCJ_t - \beta \cdot UEC_t$ |
| Stationarity | ADF test on the spread (constant, up to 1 lag chosen by AIC); half-life from an AR(1) fit |
| Trading rule | Rolling mean of the spread; long spread below the lower band, short above the upper band, exit at the mean |
| Return | Spread P&L divided by the gross value of both legs; position applied with a one-day lag |
| Metrics | Cumulative return, annualised Sharpe ratio $\sqrt{252}\,\bar r/\sigma_r$ (no risk-free rate), maximum drawdown from a high-water mark floored at 1 |

## 5. Setup and Data

In [ ]:
import os, glob, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.api import OLS
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 6), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---- Project parameters ----
DATA_DIR   = "."
URANIUM    = ["ccj", "uec", "uuuu", "dnn", "nxe", "ura"]
CONTROLS   = ["arkk", "zm"]          # non-uranium control names
PAIR       = ("ccj", "uec")          # selected pair
HEDGE_WIN  = 100                     # observations used to estimate the hedge ratio
LOOKBACK   = 5                       # rolling window of the original strategy
BAND_MULT  = 2                       # band multiplier of the original strategy
COST       = 0.0005                  # 5 bp per position change (robustness check)

In [ ]:
def load(ticker):
    """Read a Barchart daily CSV, drop the footer line and return the closing ('Last') price."""
    path = glob.glob(os.path.join(DATA_DIR, f"{ticker}_daily_historical-data-*.csv"))[0]
    d = pd.read_csv(path, engine="python")
    d["Time"] = pd.to_datetime(d["Time"], format="%m/%d/%Y", errors="coerce")
    d = d.dropna(subset=["Time"]).set_index("Time").sort_index()
    return pd.to_numeric(d["Last"], errors="coerce").rename(ticker)

prices = pd.concat([load(t) for t in URANIUM + CONTROLS], axis=1)
print("Observations per ticker:")
print(prices.count().to_string())
print(f"\nSample: {prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}")
prices.tail()

### 5.1 The selected pair

In [ ]:
P = prices[list(PAIR)].dropna()
ret = P.pct_change().dropna()
print(f"Aligned observations: {len(P)}")
print(f"Annualised volatility:\n{(ret.std() * np.sqrt(252)).round(3).to_string()}")
print(f"Daily return correlation: {ret.corr().iloc[0, 1]:.3f}")

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
(P / P.iloc[0] * 100).plot(ax=ax[0], logy=True, lw=1.2)
ax[0].set_ylabel("Rebased to 100 (log scale)"); ax[0].set_title("CCJ and UEC price history", loc="left")
(P.iloc[:, 0] / P.iloc[:, 1]).plot(ax=ax[1], color="grey", lw=1)
ax[1].set_ylabel("CCJ / UEC ratio"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

## 6. Pair Selection

Every pair is tested for cointegration with the Engle-Granger test (`statsmodels.tsa.stattools.coint`). A more negative t-statistic is stronger evidence; the pair is cointegrated at the 1% level if the statistic is below the 1% critical value. The non-uranium controls (ARKK, ZM) should **not** show cointegration with uranium names, which checks that the test does not flag everything.

In [ ]:
rows = []
for a, b in itertools.combinations(URANIUM + CONTROLS, 2):
    d = prices[[a, b]].dropna()
    if len(d) < 500:
        continue
    r = d.pct_change().dropna()
    t1, p1, crit = coint(d[a], d[b])
    t2, p2, _    = coint(d[b], d[a])
    rows.append({"pair": f"{a.upper()} / {b.upper()}", "obs": len(d),
                 "return_corr": r[a].corr(r[b]),
                 "EG_t": min(t1, t2), "p_value": min(p1, p2),
                 "crit_1pct": crit[0], "crit_5pct": crit[1],
                 "control": (a in CONTROLS) or (b in CONTROLS)})
screen = pd.DataFrame(rows).sort_values("EG_t").reset_index(drop=True)
screen["cointegrated_1pct"] = screen.EG_t < screen.crit_1pct
screen["cointegrated_5pct"] = screen.EG_t < screen.crit_5pct
display(screen.drop(columns=["crit_5pct"]).round(3).head(14))

In [ ]:
top = screen.head(14).iloc[::-1]
colors = ["#1f5fbf" if p == "CCJ / UEC" else ("#777777" if c else "#a9b7c9") for p, c in zip(top.pair, top.control)]
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top.pair, top.EG_t.abs(), color=colors)
ax.axvline(abs(screen.crit_1pct.iloc[0]), color="crimson", ls="--", lw=1, label="1% critical value")
ax.axvline(abs(screen.crit_5pct.iloc[0]), color="darkorange", ls="--", lw=1, label="5% critical value")
ax.set_xlabel("|Engle-Granger t-statistic|"); ax.grid(axis="y", alpha=0)
ax.set_title("Pair screen (blue = selected pair, dark grey = pairs involving a control)", loc="left")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

n_pairs = len(screen)
n1, n5 = int(screen.cointegrated_1pct.sum()), int(screen.cointegrated_5pct.sum())
ctrl = screen[screen.control]
print(f"{n_pairs} pairs tested: {n1} cointegrated at 1%, {n5} at 5%.")
print(f"Pairs involving a control name: {len(ctrl)}; cointegrated at 5%: {int(ctrl.cointegrated_5pct.sum())}.")

### 6.1 Why CCJ / UEC

The statistically strongest pairs involve DNN and NXE (development-stage companies), and URA (a fund that itself holds the miners). DNN also trades at very low prices, which makes execution costs and borrowing a bigger issue, and a stock-versus-ETF pair is not a like-for-like business comparison. CCJ and UEC are both liquid **operating** uranium companies with the same commodity exposure but different size and volatility, which matches the premise of the project (similar operations, priced differently). CCJ/UEC also passes the cointegration test comfortably (see table above). The high return correlation between CCJ and URA combined with weak cointegration also shows that correlation alone is not a pairs-trading criterion.

## 7. Hedge Ratio and Spread

The hedge ratio is the OLS slope of CCJ on UEC over the first 100 observations (no intercept). The ratio is then held fixed, so the remaining history is traded with a hedge ratio that was known on day 100.

In [ ]:
df = P.copy(); df.columns = ["ccj", "uec"]
ols_fit = OLS(df.ccj.iloc[:HEDGE_WIN], df.uec.iloc[:HEDGE_WIN]).fit()
beta = float(ols_fit.params.iloc[0])
df["spread"] = df.ccj - beta * df.uec
print(f"Hedge ratio (beta): {beta:.4f} UEC shares per CCJ share")

ax = df.spread.plot(lw=0.9, label=f"Spread = CCJ - {beta:.2f} x UEC")
ax.axhline(df.spread.mean(), color="crimson", ls="--", lw=1, label=f"Mean ({df.spread.mean():.2f})")
ax.axhline(0, color="grey", lw=0.8); ax.set_ylabel("Spread (USD)"); ax.legend(frameon=False)
ax.set_title("Hedged spread", loc="left"); plt.show()

### 7.1 Stationarity (ADF) and half-life

In [ ]:
adf = adfuller(df.spread, maxlag=1)
adf_t, adf_p, adf_crit = adf[0], adf[1], adf[4]
print(f"ADF t-statistic : {adf_t:.3f}")
print(f"Critical values : 1% {adf_crit['1%']:.3f} | 5% {adf_crit['5%']:.3f} | 10% {adf_crit['10%']:.3f}")
print("Unit root rejected at 1%: spread is treated as stationary." if adf_t < adf_crit["1%"]
      else "Unit root NOT rejected at 1%.")

# Half-life of mean reversion from an AR(1) fit: dS_t = a + b * S_{t-1} + e
y = df.spread.values
b = np.polyfit(y[:-1], np.diff(y), 1)[0]
rho = 1 + b
half_life = -np.log(2) / np.log(rho) if 0 < rho < 1 else np.inf
print(f"AR(1) coefficient: {rho:.4f}  ->  half-life of about {half_life:.0f} trading days")

In [ ]:
# Stability: ADF and level of the spread by calendar year
yearly = []
for yr, g in df.spread.groupby(df.index.year):
    yearly.append({"year": yr, "ADF_t": adfuller(g, maxlag=1)[0], "mean": g.mean(), "std": g.std()})
yearly = pd.DataFrame(yearly).set_index("year")
display(yearly.round(2))
n_sig = int((yearly.ADF_t < adf_crit["5%"]).sum())

## 8. Trading Strategy and Backtest

### 8.1 Strategy as implemented

The rule below reproduces the project's original script exactly. A rolling mean of the spread is computed over `LOOKBACK` days. **The upper and lower bands are built as `mean ± BAND_MULT × mean`** (the multiplier is applied to the rolling mean, as in the script, not to the rolling standard deviation). Long entries occur when the spread falls below the lower band, short entries above the upper band, and both exit when the spread returns to the rolling mean. Returns are the daily change in the spread divided by the gross value of both legs, with the position lagged by one day.

In [ ]:
def reversion(data, lookback, mult):
    data = data.copy()
    data["mvg_avg"]     = data.spread.rolling(window=lookback).mean()
    data["mvg_std_dev"] = data.spread.rolling(window=lookback).std()
    data["upper"] = data.mvg_avg + mult * data.mvg_avg       # as in the original script
    data["lower"] = data.mvg_avg - mult * data.mvg_avg

    data["long_entry"]  = data.spread < data.lower
    data["long_exit"]   = data.spread >= data.mvg_avg
    data["long_positions"] = np.nan
    data.loc[data.long_entry,  "long_positions"] = 1
    data.loc[data.long_exit,   "long_positions"] = 0
    data["long_positions"] = data.long_positions.ffill()

    data["short_entry"] = data.spread > data.upper
    data["short_exit"]  = data.spread <= data.mvg_avg
    data["short_positions"] = np.nan
    data.loc[data.short_entry, "short_positions"] = -1
    data.loc[data.short_exit,  "short_positions"] = 0
    data["short_positions"] = data.short_positions.ffill()

    data["positions"] = data.long_positions + data.short_positions
    return data

def add_returns(data, beta):
    data = data.copy()
    data["percentage_change"] = (data.spread - data.spread.shift(1)) / (beta * data.uec + data.ccj)
    data["strategy_returns"]  = data.positions.shift(1) * data.percentage_change
    data["cumulative_returns"] = (data.strategy_returns + 1).cumprod()
    return data.dropna()

def drawdown_calculation(cum):
    running_max = np.maximum.accumulate(cum.dropna())
    running_max[running_max < 1] = 1
    return cum / running_max - 1

def perf(r):
    """Performance summary from a series of daily strategy returns."""
    r = r.dropna(); cum = (1 + r).cumprod(); years = len(r) / 252
    return {"Growth of 1 unit": cum.iloc[-1],
            "CAGR": cum.iloc[-1] ** (1 / years) - 1,
            "Annual vol": r.std() * np.sqrt(252),
            "Sharpe": r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
            "Max drawdown": drawdown_calculation(cum).min()}

orig = add_returns(reversion(df, LOOKBACK, BAND_MULT), beta)
orig_perf = perf(orig.strategy_returns)
turn = orig.positions.diff().abs().fillna(0)
orig_cost_perf = perf(orig.strategy_returns - COST * turn)

pos_lag = orig.positions.shift(1)
n_long, n_short, n_flat = int((pos_lag == 1).sum()), int((pos_lag == -1).sum()), int((pos_lag == 0).sum())
n_changes = int((orig.positions.diff().abs() > 0).sum())

# Buy and hold CCJ over the same dates
bh_ret = df.ccj.pct_change().loc[orig.index[0]:]
bh_perf = perf(bh_ret)

summary = pd.DataFrame({"Strategy as implemented": orig_perf,
                        "Same, 5 bp cost per position change": orig_cost_perf,
                        "CCJ buy and hold": bh_perf}).T
display(summary.round(3))
print(f"Days long spread: {n_long} | short spread: {n_short} | flat: {n_flat} | position changes: {n_changes}")

# Optional check against the saved results file from the original script
res_path = os.path.join(DATA_DIR, "results.csv")
if os.path.exists(res_path):
    saved = pd.read_csv(res_path, index_col=0, parse_dates=True)
    print("Matches saved results.csv:", np.allclose(saved.cumulative_returns.values, orig.cumulative_returns.values))

In [ ]:
cum_orig = orig.cumulative_returns
cum_cost = (1 + orig.strategy_returns - COST * turn).cumprod()
cum_bh   = (1 + bh_ret.fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(12, 5.5))
cum_orig.plot(ax=ax, lw=1.4, label="Strategy as implemented")
cum_cost.plot(ax=ax, lw=1.1, ls="--", label="With 5 bp cost")
cum_bh.plot(ax=ax, lw=1, color="grey", label="CCJ buy and hold")
ax.set_yscale("log"); ax.axhline(1, color="k", lw=0.6)
ax.set_ylabel("Growth of 1 unit (log scale)"); ax.set_title("Cumulative returns", loc="left"); ax.legend(frameon=False)
plt.show()

dd = drawdown_calculation(orig.cumulative_returns)
plt.figure(figsize=(12, 4))
dd.mul(100).plot(color="crimson"); plt.fill_between(dd.index, dd.mul(100), color="crimson", alpha=0.4)
plt.ylabel("Drawdown (%)"); plt.title(f"Drawdown (maximum {dd.min()*100:.2f}%, on {dd.idxmin():%d %b %Y})", loc="left")
plt.show()

by_year = orig.strategy_returns.groupby(orig.index.year).apply(lambda r: (1 + r).prod() - 1)
display(by_year.to_frame("Strategy return").T.style.format("{:.1%}") if hasattr(by_year, "to_frame") else by_year)

### 8.2 Robustness and critical review

Three checks assess how much of the result depends on implementation choices rather than on the spread itself.

**(a) Band construction.** Because the bands are multiples of the rolling *mean*, their width depends on the level of the spread and not on its volatility. Below, the average half-width of the implemented bands is compared with that of a true two-standard-deviation band, and a standard z-score version is backtested. A z-score computed on 5 observations can never exceed $(n-1)/\sqrt{n} = 1.79$, so a 2-sigma rule cannot trigger with the 5-day window; a 20-day window is used instead.

In [ ]:
band_orig = (BAND_MULT * orig.mvg_avg.abs()).mean()
band_std  = (BAND_MULT * orig.mvg_std_dev).mean()
max_z_5   = (LOOKBACK - 1) / np.sqrt(LOOKBACK)
print(f"Average band half-width, implemented rule : {band_orig:.2f}")
print(f"Average band half-width, true 2-sigma band : {band_std:.2f}")
print(f"Largest possible z-score with a {LOOKBACK}-day window: {max_z_5:.2f}")

def zscore_strategy(data, beta, const, lookback, z_in, z_out=0.0, start=0):
    """Standard z-score mean reversion on spread = ccj - beta*uec - const; returns the frame with returns."""
    d = data[["ccj", "uec"]].copy()
    d["spread"] = d.ccj - beta * d.uec - const
    z = (d.spread - d.spread.rolling(lookback).mean()) / d.spread.rolling(lookback).std()
    pos = np.zeros(len(d)); cur = 0
    for i in range(len(d)):
        zi = z.iloc[i]
        if i < start or np.isnan(zi):
            cur = 0; continue
        if cur == 0:
            cur = -1 if zi > z_in else (1 if zi < -z_in else 0)
        elif cur == 1 and zi >= -z_out:
            cur = 0
        elif cur == -1 and zi <= z_out:
            cur = 0
        pos[i] = cur
    d["pos"] = pos
    gross = (d.ccj + beta * d.uec).shift(1)
    d["gross_ret"] = d.pos.shift(1) * (d.ccj.diff() - beta * d.uec.diff()) / gross
    d["turn"] = d.pos.diff().abs().fillna(0)
    d["net_ret"] = d.gross_ret - COST * d.turn
    return d

zb = zscore_strategy(df, beta, 0.0, 20, 2.0)
band_test = pd.DataFrame({"Implemented rule (gross)": orig_perf,
                          "z-score rule, 20-day, 2 sigma (gross)": perf(zb.gross_ret),
                          "z-score rule, 20-day, 2 sigma (5 bp cost)": perf(zb.net_ret)}).T
display(band_test.round(3))

**(b) Walk-forward test.** The hedge ratio is re-estimated (with an intercept) on the first 1,000 observations, then held fixed and traded out of sample with a 20-day z-score rule (enter at 2 sigma, exit at 0). If the cointegrating relationship were stable, the spread built from the training window should remain stationary after it.

In [ ]:
SPLIT = 1000
train, test = df.iloc[:SPLIT], df.iloc[SPLIT:]
Xtr = np.column_stack([train.uec.values, np.ones(SPLIT)])
b_wf, c_wf = np.linalg.lstsq(Xtr, train.ccj.values, rcond=None)[0]

train_t = coint(train.ccj, train.uec)[0]
sp_oos = test.ccj - b_wf * test.uec - c_wf
oos_adf = adfuller(sp_oos, maxlag=1)
print(f"Training window: {train.index[0]:%d %b %Y} to {train.index[-1]:%d %b %Y}")
print(f"Hedge ratio {b_wf:.3f}, intercept {c_wf:.2f}  (full-sample original ratio: {beta:.3f})")
print(f"In-sample Engle-Granger t = {train_t:.2f}; out-of-sample ADF t on the spread = {oos_adf[0]:.2f} (5% critical {oos_adf[4]['5%']:.2f})")

wf = zscore_strategy(df, b_wf, c_wf, 20, 2.0, start=SPLIT).iloc[SPLIT:]
wf_perf_g, wf_perf_n = perf(wf.gross_ret), perf(wf.net_ret)
bh_oos = perf(test.ccj.pct_change())
wf_table = pd.DataFrame({"Walk-forward (gross)": wf_perf_g, "Walk-forward (5 bp cost)": wf_perf_n,
                         "CCJ buy and hold, same period": bh_oos}).T
display(wf_table.round(3))
print(f"Position changes: {int((wf.pos.diff().abs() > 0).sum())}")

**(c) Parameter sensitivity.** The walk-forward rule is repeated for a grid of lookback windows and entry thresholds to see whether the result depends on one lucky parameter choice.

In [ ]:
grid = []
for lb in [10, 20, 40, 60]:
    for z_in in [1.5, 2.0, 2.5]:
        g = zscore_strategy(df, b_wf, c_wf, lb, z_in, start=SPLIT).iloc[SPLIT:]
        p = perf(g.net_ret)
        grid.append({"lookback": lb, "entry z": z_in, "Sharpe": p["Sharpe"], "Growth": p["Growth of 1 unit"], "Max DD": p["Max drawdown"]})
grid = pd.DataFrame(grid)
display(grid.pivot(index="lookback", columns="entry z", values="Sharpe").round(2).rename_axis(columns="Sharpe (net), entry z"))

## 9. Findings

In [ ]:
tstr = "stationary (unit root rejected at 1%)" if adf_t < adf_crit["1%"] else "not clearly stationary"
display(Markdown(f'''
1. **Pair selection.** Of {n_pairs} pairs tested, {n1} are cointegrated at 1% and {n5} at 5%; of the {len(ctrl)} pairs that include a non-uranium control, {int(ctrl.cointegrated_5pct.sum())} are cointegrated at 5%. CCJ / UEC has an Engle-Granger statistic of {screen.loc[screen.pair == "CCJ / UEC", "EG_t"].iloc[0]:.2f} against a 1% critical value of {screen.crit_1pct.iloc[0]:.2f}.

2. **Spread.** With a hedge ratio of {beta:.2f} UEC shares per CCJ share, the full-sample spread is {tstr}: ADF t = {adf_t:.2f} against {adf_crit["1%"]:.2f}. The half-life of reversion is about {half_life:.0f} trading days. Tested year by year, only {n_sig} of {len(yearly)} years pass the 5% ADF critical value, and the yearly mean of the spread ranges from {yearly["mean"].min():.1f} to {yearly["mean"].max():.1f}, so the stationarity rests on long swings around a mean that shifts.

3. **Strategy as implemented.** One unit of capital grew to **{orig_perf["Growth of 1 unit"]:.2f}** ({orig_perf["Growth of 1 unit"]-1:+.0%}) from {orig.index[0]:%b %Y} to {orig.index[-1]:%b %Y}, with an annualised Sharpe ratio of **{orig_perf["Sharpe"]:.2f}**, annual volatility of {orig_perf["Annual vol"]:.1%} and a maximum drawdown of **{orig_perf["Max drawdown"]:.1%}**. CCJ buy and hold gave {bh_perf["Growth of 1 unit"]:.2f}x with a {bh_perf["Max drawdown"]:.1%} drawdown. The strategy was in a position on {n_long + n_short} of {n_long + n_short + n_flat} days ({n_long} long, {n_short} short) and changed position {n_changes} times.

4. **Costs.** A cost of 5 bp per position change lowers growth to {orig_cost_perf["Growth of 1 unit"]:.2f}x and the Sharpe ratio to {orig_cost_perf["Sharpe"]:.2f}.

5. **Band construction.** The implemented bands have an average half-width of {band_orig:.2f}, versus {band_std:.2f} for a true two-sigma band, so entries depend on the level of the spread rather than on how unusual a deviation is. A standard 20-day z-score rule on the same spread gives {perf(zb.gross_ret)["Growth of 1 unit"]:.2f}x and a Sharpe ratio of {perf(zb.gross_ret)["Sharpe"]:.2f} before costs.

6. **Walk-forward.** A hedge ratio fitted on {train.index[0]:%Y} to {train.index[-1]:%Y} came out at {b_wf:.2f} (not {beta:.2f}); the in-sample cointegration statistic was {train_t:.2f} and the out-of-sample spread had an ADF statistic of {oos_adf[0]:.2f} (5% critical {oos_adf[4]["5%"]:.2f}). Traded out of sample, the z-score rule returned {wf_perf_n["Growth of 1 unit"]:.2f}x after costs with a Sharpe ratio of {wf_perf_n["Sharpe"]:.2f} and a maximum drawdown of {wf_perf_n["Max drawdown"]:.1%}. Across the parameter grid the net Sharpe ratio ranged from {grid.Sharpe.min():.2f} to {grid.Sharpe.max():.2f}.
'''))

## 10. Conclusion

CCJ and UEC pass both the economic and the statistical screen for a uranium pairs trade: the pair is cointegrated at the 1% level over the full sample, and the spread built from an early OLS hedge ratio is stationary with a half-life of roughly six to eight weeks. The strategy as implemented in the project produced a strong backtest with a drawdown well below that of simply holding CCJ, which confirms that the idea is worth pursuing.

The robustness checks, however, show that the result should be read with care. It depends on the band formulation, it falls after transaction costs, and in the walk-forward test the hedge ratio estimated on the early data did not remain valid, so the out-of-sample results are weaker. A realistic expectation is therefore a modest Sharpe ratio rather than the in-sample figure, and the hedge-ratio instability is the main risk to the strategy.

## 11. Limitations and Further Work

- Use standard-deviation-based bands with a longer window and choose parameters on a training period, evaluating them on a separate test period.
- Re-estimate the hedge ratio on a rolling window or with a Kalman filter, and work with log prices.
- Add borrowing and slippage costs, a stop-loss on divergence, and position sizing by spread volatility.
- Test the other cointegrated pairs and a basket (for example CCJ against a UEC, UUUU and NXE portfolio) to reduce single-name risk.
- Correct for having tested 28 pairs, and check sensitivity to the critical values used.

*This notebook is for educational purposes and is not investment advice.*

## 12. References

- Engle, R. F. and Granger, C. W. J. (1987). Co-integration and error correction: representation, estimation and testing. *Econometrica*, 55(2).
- Dickey, D. A. and Fuller, W. A. (1979). Distribution of the estimators for autoregressive time series with a unit root. *Journal of the American Statistical Association*, 74.
- MacKinnon, J. G. (2010). Critical values for cointegration tests. Queen's Economics Department Working Paper No. 1227.
- Gatev, E., Goetzmann, W. N. and Rouwenhorst, K. G. (2006). Pairs trading: performance of a relative-value arbitrage rule. *Review of Financial Studies*, 19(3).
- Vidyamurthy, G. (2004). *Pairs Trading: Quantitative Methods and Analysis*. Wiley.
- Chan, E. P. (2013). *Algorithmic Trading: Winning Strategies and Their Rationale*. Wiley.
- Data: Barchart.com daily historical prices. Original script: `statistical_arbitrage_uranium.py`.